# CMT large300 — Comment la friction de surface se transmet à la troposphère
### (version 2 — corrigée d'après les résultats du run précédent)

Ce notebook part de l'observation du panneau $\bar u(z,t)$ : **entre 2 et 8 km, le vent moyen
présente un déphasage vertical qui descend au fil du temps** — une advection / onde. On relie ce
fait au flux de Reynolds, au bilan de quantité de mouvement (Navier–Stokes moyenné) et, au bout,
au **sujet du stage** : *comment la friction exercée à la surface est communiquée à toute la
troposphère par la turbulence convective* (la viscosité moléculaire étant négligeable).

**Corrections majeures apportées à la version précédente** (diagnostiquées sur ses sorties) :

1. **Pas de temps.** La version 1 lisait $\Delta t = 1$ s (coordonnée temporelle en indices) →
   toutes les tendances et vitesses étaient fausses d'un facteur $\sim$21600. Le protocole RCEMIP
   impose des snapshots 3D **toutes les 6 h** : on fixe $\Delta t = 21600$ s (détection robuste +
   garde-fou). Cela remet $\partial_t\bar u$, $c_z$ et le bilan aux bonnes échelles.
2. **Bilan de qdm.** La v1 testait $\partial_t\bar u = -\rho_0^{-1}\partial_z(\rho_0\overline{u'w'})$
   seul → NSE $=0$ (faux : il manque l'advection et surtout le **forçage grande échelle** propre au
   RCE). On écrit le **bilan complet**, on compare les amplitudes des termes, et on identifie le
   résidu au forçage imposé — honnêtement.
3. **Descente de phase.** $c_z$ était donné en dizaines de m/s (absurde). Corrigé en **cm/s**
   (échelle physique), avec distribution + test de robustesse : sur 34 pas, on annonce ce que la
   statistique permet, sans surinterpréter.
4. **Modèle.** La v1 donnait NSE $=-1.2$ (fuite de normalisation, fit point-à-point sur trop peu de
   points). On fit désormais le **profil moyen** $\overline{u'w'}(z)$ avec des coefficients physiques
   directs, et on montre *pourquoi* le diffusif échoue (régime contre-gradient).

**Cadre biblio.** Zhang & Wu (2003), Romps (2012), Moncrieff (1992) / Dixit et al. (2021) pour le
CMT ; Holton–Lindzen (1972), Plumb (1977) pour la descente de phase (mécanisme QBO).

> Stack `numpy/scipy/xarray/matplotlib`, lecture NetCDF niveau par niveau, fenêtre stationnaire =
> dernier tiers temporel.

## 0. Configuration — avec correction du pas de temps

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os
from scipy.signal import detrend, hilbert

plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.grid':True,
                     'grid.alpha':0.25,'image.cmap':'RdBu_r'})

# ============================================================
#  CONFIGURATION large300
# ============================================================
DIR_3D='3D'; DIR_2D='2D'; DIR_1D='1D'
def path3d(v): return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{v}.nc')
def path2d(v): return os.path.join(DIR_2D, f'MESONH_RCE_large300_2D_{v}.nc')
def path1d(v): return os.path.join(DIR_1D, f'MESONH_RCE_large300_1D_{v}.nc')

Rd,Rv = 287.05,461.5; EPSILON=Rd/Rv; g=9.81
Z_TOP_KM = 15.0

_ds=xr.open_dataset(path3d('ua')); _da=_ds['ua']
dim_t,dim_z,dim_y,dim_x=_da.dims
n_t=_da.sizes[dim_t]; n_z=_da.sizes[dim_z]; n_y=_da.sizes[dim_y]; n_x=_da.sizes[dim_x]
_ds.close(); del _ds,_da; gc.collect()

_t1=xr.open_dataset(path1d('ua_avg')); alt=_t1['altitude'].values.astype(float).copy(); _t1.close()

t_stat=int(2*n_t/3); idx_stat=slice(t_stat,None); n_stat=n_t-t_stat
zkm=alt/1000.0; mask_show=alt<=Z_TOP_KM*1000
m28=(alt>=2000)&(alt<=8000)

# -----------------------------------------------------------------
#  PAS DE TEMPS PHYSIQUE  (CORRECTION MAJEURE)
#  La coord temporelle du fichier est en INDICES (0,1,2,...), donc
#  diff=1 "s" est FAUX. Le protocole RCEMIP prescrit des snapshots
#  3D toutes les 6 h -> dt_phys = 21600 s. On tente de lire un vrai
#  dt (secondes ou datetime) et sinon on impose 6 h.
# -----------------------------------------------------------------
DT_RCEMIP_3D = 6*3600.0     # 6 h (protocole RCEMIP pour les sorties 3D instantanees)
dt_phys = DT_RCEMIP_3D
_src = 'defaut RCEMIP (6 h)'
try:
    _dst=xr.open_dataset(path3d('ua')); _tc=_dst[dim_t].values; _dst.close()
    if np.issubdtype(np.asarray(_tc).dtype,'datetime64'):
        d=np.median(np.diff(_tc[t_stat:].astype('datetime64[s]').astype(float)))
        if np.isfinite(d) and d>60: dt_phys=float(d); _src='coord datetime du fichier'
    else:
        d=np.median(np.diff(np.asarray(_tc,float)[t_stat:]))
        # accepte seulement si ca ressemble a des secondes plausibles (>60 s)
        if np.isfinite(d) and d>60: dt_phys=float(d); _src='coord numerique du fichier'
except Exception:
    pass
dx=1000.0; dy=dx; x=np.arange(n_x)*dx

print(f'Grille : {n_t} t x {n_z} z x {n_y} y x {n_x} x   |  dx=dy={dx:.0f} m')
print(f'Altitude : {alt[0]:.0f} -> {alt[-1]:.0f} m')
print(f'Stationnaire : t={t_stat}->{n_t-1} ({n_stat} pas)')
print(f'dt_phys = {dt_phys:.0f} s = {dt_phys/3600:.1f} h   [source: {_src}]')
print(f'Duree fenetre analysee : {n_stat*dt_phys/86400:.1f} jours')
if dt_phys < 60:
    print('  ATTENTION : dt suspect (<60s). Verifie la coord temporelle du fichier !')


In [ ]:
# ============================================================
#  Champs de base : rho0(z), ubar(z,t), wbar(z,t), flux(z,t)
#  + forcage grande echelle du vent si disponible (RCEMIP : relaxation)
#  lecture niveau par niveau (RAM)
# ============================================================
rho0=np.zeros(n_z)
ds_ta=xr.open_dataset(path3d('ta')); ds_pa=xr.open_dataset(path3d('pa')); ds_hus=xr.open_dataset(path3d('hus'))
for iz in range(n_z):
    ta=ds_ta['ta'].isel({dim_z:iz,dim_t:idx_stat}).values
    pa=ds_pa['pa'].isel({dim_z:iz,dim_t:idx_stat}).values
    hus=ds_hus['hus'].isel({dim_z:iz,dim_t:idx_stat}).values
    tv=ta*(1.0+(1.0/EPSILON-1.0)*hus)
    rho0[iz]=np.mean(pa/(Rd*tv))
ds_ta.close(); ds_pa.close(); ds_hus.close(); gc.collect()

flux=np.zeros((n_z,n_stat)); ubar=np.zeros((n_z,n_stat)); wbar=np.zeros((n_z,n_stat))
uw_var=np.zeros((n_z,n_stat))    # variance pour diagnostic
ds_u=xr.open_dataset(path3d('ua')); ds_w=xr.open_dataset(path3d('wa'))
for iz in range(n_z):
    u=ds_u['ua'].isel({dim_z:iz,dim_t:idx_stat}).values
    w=ds_w['wa'].isel({dim_z:iz,dim_t:idx_stat}).values
    ub=u.mean(axis=(1,2)); wb=w.mean(axis=(1,2))
    up=u-ub[:,None,None]; wp=w-wb[:,None,None]
    flux[iz]=rho0[iz]*(up*wp).mean(axis=(1,2))
    ubar[iz]=ub; wbar[iz]=wb
ds_u.close(); ds_w.close(); gc.collect()

dudz=np.gradient(ubar,alt,axis=0); d2udz2=np.gradient(dudz,alt,axis=0)
uw=flux/rho0[:,None]

# amplitude typique des champs pour se reperer
print('Champs prets :', flux.shape)
print(f'  ubar   : [{ubar[mask_show].min():+.2f}, {ubar[mask_show].max():+.2f}] m/s')
print(f'  flux   : [{flux[mask_show].min():+.4f}, {flux[mask_show].max():+.4f}] Pa')
print(f'  contrainte de surface tau_s = flux(z=0) = {flux[0].mean():+.4f} Pa (moyenne temps)')


## 1. La figure de départ

Cinq diagnostics superposés (couleur = temps). Le panneau $\bar u(z)$, bande 2–8 km surlignée :
les extrema migrent vers le bas au fil du temps.

In [ ]:
# ============================================================
#  §1 — La figure de depart (5 diagnostics, couleur = temps)
# ============================================================
its=np.linspace(0,n_stat-1,10,dtype=int)
cmap=plt.get_cmap('viridis'); colors=cmap(np.linspace(0,1,len(its)))
fig,axes=plt.subplots(1,5,figsize=(17,5),sharey=True)
for ax,(fld,lab) in zip(axes,[(flux,r"$\rho_0\langle u'w'\rangle$"),(dudz,r"$\partial_z\bar u$"),
        (ubar,r"$\bar u$ (m/s)"),(wbar,r"$\bar w$ (m/s)"),(d2udz2,r"$\partial_z^2\bar u$")]):
    for it,c in zip(its,colors): ax.plot(fld[mask_show,it],zkm[mask_show],lw=1.2,color=c)
    ax.axvline(0,color='k',lw=.5); ax.set_xlabel(lab)
axes[2].axhspan(2,8,color='red',alpha=.07)
axes[0].set_ylabel('z (km)'); axes[0].set_ylim(0,Z_TOP_KM)
sm=plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(t_stat,t_stat+n_stat-1))
fig.colorbar(sm,ax=axes,label='indice temporel',pad=.01,aspect=30)
fig.suptitle('Diagnostics 0-15 km (bande rouge = 2-8 km : le dephasage a expliquer)')
plt.show()
print("Panneau ubar(z) : entre 2 et 8 km, les extrema migrent vers le bas au fil du temps.")


## 2. Quantifier la descente de phase

Hovmöller $\bar u'(z,t)$ (anomalie, lissée en $z$ pour réduire le bruit de maille) + mesure de la
vitesse de phase verticale $c_z=-(\partial_t\varphi)/(\partial_z\varphi)$ par phase de Hilbert,
**en cm/s**. On donne la distribution complète et la fraction de points en descente, plutôt qu'une
valeur unique — 34 pas de temps ($\sim$8 jours) est une fenêtre courte, on reste honnête sur ce
qu'elle permet d'affirmer.

In [ ]:
# ============================================================
#  §2 — DESCENTE DE PHASE : Hovmoller + mesure de c_z
#  Corrections vs version precedente :
#   - dt_phys = 6 h (et non 1 s) -> c_z en unites PHYSIQUES correctes
#   - filtrage : on isole le mode dominant en lissant legerement en z
#   - on donne des INTERVALLES (percentiles) et un test de robustesse,
#     au lieu d'une valeur unique potentiellement bruitee (34 pas seulement)
# ============================================================
uprime=ubar-ubar.mean(axis=1,keepdims=True)
# lissage vertical doux (3 pts) pour attenuer le bruit de maille avant analyse de phase
from scipy.ndimage import uniform_filter1d
uprime_s=uniform_filter1d(uprime,size=3,axis=0,mode='nearest')
tt=np.arange(n_stat)

fig,axes=plt.subplots(1,2,figsize=(13,5))
ax=axes[0]; ax.grid(False)
v=np.percentile(np.abs(uprime_s[mask_show]),98)+1e-9
im=ax.pcolormesh(tt+t_stat,zkm[mask_show],uprime_s[mask_show],cmap='RdBu_r',
                 norm=TwoSlopeNorm(0,-v,v),shading='auto')
ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('t (indice global)'); ax.set_ylabel('z (km)')
ax.set_title(r"Hovmoller $\bar u'(z,t)$ (bandes inclinees = descente)")
fig.colorbar(im,ax=ax,pad=.02,label='m/s')

# --- c_z par phase de Hilbert, en unites physiques, avec distribution ---
analytic=hilbert(detrend(uprime_s,axis=1),axis=1)
phase=np.unwrap(np.angle(analytic),axis=1)
dphi_dt=np.gradient(phase,dt_phys,axis=1)   # rad/s
dphi_dz=np.gradient(phase,alt,axis=0)       # rad/m
with np.errstate(divide='ignore',invalid='ignore'):
    cz=-dphi_dt/dphi_dz                       # m/s
cz_28=cz[m28]; cz_28=cz_28[np.isfinite(cz_28)]
# on garde les valeurs physiquement plausibles (|c_z|<1 m/s : ondes lentes en RCE)
cz_ok=cz_28[np.abs(cz_28)<1.0]
q25,q50,q75=np.percentile(cz_ok,[25,50,75]) if cz_ok.size else (np.nan,)*3
frac_desc=np.mean(cz_ok<0) if cz_ok.size else np.nan

ax=axes[1]
ax.hist(cz_ok*100,bins=30,color='steelblue',alpha=.85)
ax.axvline(q50*100,color='r',lw=2,label=f'mediane={q50*100:+.2f} cm/s')
ax.axvline(0,color='k',lw=.8)
ax.set_xlabel('vitesse de phase verticale c_z (cm/s)')
ax.set_ylabel('nb de points (z,t) dans 2-8 km')
ax.set_title(f'Distribution de c_z\n{frac_desc:.0%} des points DESCENDENT (c_z<0)')
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

# periode et longueur d'onde dominantes (2-8 km)
om=np.abs(dphi_dt[m28]); om=om[np.isfinite(om)&(om>0)]
kz=np.abs(dphi_dz[m28]); kz=kz[np.isfinite(kz)&(kz>0)]
T_h=2*np.pi/np.median(om)/3600 if om.size else np.nan
lam=2*np.pi/np.median(kz)/1000 if kz.size else np.nan
print(f'c_z (2-8 km) : mediane={q50*100:+.2f} cm/s  [IQR {q25*100:+.2f}, {q75*100:+.2f}]')
print(f'  fraction descendante : {frac_desc:.0%}')
print(f'  periode dominante ~ {T_h:.1f} h ({T_h/24:.1f} j)   |  lambda_z ~ {lam:.1f} km')
print()
if frac_desc>0.6:
    print('=> Majorite des points en DESCENTE : signal coherent avec une onde qui')
    print('   transporte du momentum vers le haut. (A confirmer sur run plus long :')
    print(f'   {n_stat} pas ~ {n_stat*dt_phys/86400:.0f} j est court pour la statistique spectrale.)')
else:
    print('=> Signal MIXTE (montee+descente) sur cette fenetre courte. La descente')
    print('   visible sur le Hovmoller est locale/intermittente, pas encore un mode')
    print('   QBO-like etabli. Honnete : il faut un run plus long pour trancher.')


## 3. Bilan de quantité de mouvement (complet)

Le vrai bilan de $\bar u$ en RCE forcé :
$$\partial_t\bar u = \underbrace{-\tfrac{1}{\rho_0}\partial_z(\rho_0\overline{u'w'})}_{\text{friction convective}}
\;\underbrace{-\,\bar w\,\partial_z\bar u}_{\text{advection}}\;+\;\underbrace{F_{GE}}_{\text{forçage grande échelle}}.$$
On ne peut pas espérer que la friction convective **seule** égale $\partial_t\bar u$ (l'erreur de la
v1). On procède en trois diagnostics robustes : **(A)** amplitudes relatives des termes calculables,
**(B)** le résidu $R=\partial_t\bar u+\bar w\partial_z\bar u+\rho_0^{-1}\partial_z(\rho_0\overline{u'w'})=-F_{GE}$
comme signature du forçage, **(C)** corrélation de *structure* (anomalies) entre la descente de
$\bar u$ et la divergence du flux.

In [ ]:
# ============================================================
#  §3 — BILAN DE QUANTITE DE MOUVEMENT (version corrigee & honnete)
#
#  Bilan complet de u moyenne horizontalement :
#    d_t ubar = -1/rho0 d_z(rho0<u'w'>)  - wbar d_z ubar  + F_GE
#               \___ friction convective ___/  \_advection_/  \forcage GE/
#
#  En RCE, F_GE = relaxation grande echelle (non fournie ici). On ne peut
#  donc PAS esperer d_t ubar = friction convective seule (c'etait l'erreur :
#  NSE=0). On procede autrement, en 3 diagnostics ROBUSTES :
#
#   (A) comparer les AMPLITUDES des 3 termes calculables (friction conv.,
#       advection verticale, tendance) -> qui domine le bilan resolu ?
#   (B) le residu  R = d_t ubar + wbar d_z ubar + 1/rho0 d_z(rho0<u'w'>)
#       = -F_GE : sa structure verticale = signature du forcage impose.
#   (C) STRUCTURE (correlation de forme, pas d'amplitude) entre la descente
#       de ubar et la divergence du flux, dans la bande 2-8 km.
# ============================================================
dudt  = np.gradient(ubar,dt_phys,axis=1)              # tendance (m/s/s)
divF  = np.gradient(flux,alt,axis=0)                  # d_z(rho0<u'w'>)
fric  = -divF/rho0[:,None]                            # friction convective (m/s/s)
adv   = -wbar*dudz                                    # advection verticale resolue (m/s/s)
resid = dudt - fric - adv                             # = -F_GE (forcage grande echelle)

# unites: on affiche tout en m/s/jour
S=86400.0
def rms(a): return np.sqrt(np.mean(a**2))

# (A) amplitudes par bande
print('=== (A) Amplitude RMS des termes du bilan (m/s/jour) ===')
for lab,fld in [('d_t ubar (tendance)',dudt),('friction convective',fric),
                ('advection verticale',adv),('residu = -F_GE',resid)]:
    print(f'  {lab:24s} : 0-15km={rms(fld[mask_show])*S:6.3f}   2-8km={rms(fld[m28])*S:6.3f}')

fig,axes=plt.subplots(1,3,figsize=(16,5),sharey=True)
# panneau 1 : profils RMS temporels des 3 termes
ax=axes[0]
ax.plot(np.sqrt((fric**2).mean(1))[mask_show]*S, zkm[mask_show],'r',lw=1.8,label='friction convective')
ax.plot(np.sqrt((adv**2).mean(1))[mask_show]*S, zkm[mask_show],'g',lw=1.8,label='advection vert.')
ax.plot(np.sqrt((dudt**2).mean(1))[mask_show]*S, zkm[mask_show],'k',lw=1.8,label='tendance')
ax.plot(np.sqrt((resid**2).mean(1))[mask_show]*S, zkm[mask_show],color='grey',lw=1.2,ls='--',label='residu (-F_GE)')
ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('RMS (m/s/jour)'); ax.set_ylabel('z (km)')
ax.set_title('(A) Poids des termes du bilan'); ax.legend(fontsize=8)

# panneau 2 : Hovmoller de la friction convective (le terme convectif)
ax=axes[1]; ax.grid(False)
v=np.percentile(np.abs(fric[mask_show]),98)*S+1e-9
im=ax.pcolormesh(np.arange(n_stat)+t_stat,zkm[mask_show],fric[mask_show]*S,
                 cmap='RdBu_r',norm=TwoSlopeNorm(0,-v,v),shading='auto')
ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('t'); ax.set_title(r'(B) friction convective $-\rho_0^{-1}\partial_z(\rho_0\overline{u^\prime w^\prime})$')
fig.colorbar(im,ax=ax,pad=.02,label='m/s/jour')

# panneau 3 : residu = -F_GE
ax=axes[2]; ax.grid(False)
im=ax.pcolormesh(np.arange(n_stat)+t_stat,zkm[mask_show],resid[mask_show]*S,
                 cmap='PuOr_r',norm=TwoSlopeNorm(0,-v,v),shading='auto')
ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('t'); ax.set_title('(B) residu = forcage grande echelle')
fig.colorbar(im,ax=ax,pad=.02,label='m/s/jour')
fig.tight_layout(); plt.show()

# (C) correlation de STRUCTURE entre tendance et friction convective dans 2-8 km
#     (on retire la moyenne temporelle a chaque z -> on compare les ANOMALIES)
a=(dudt-dudt.mean(1,keepdims=True))[m28].ravel()
b=(fric-fric.mean(1,keepdims=True))[m28].ravel()
r_struct=np.corrcoef(a,b)[0,1]
# part de la tendance expliquee par friction+advection (les 2 termes resolus)
resolved=fric+adv
c=(resolved-resolved.mean(1,keepdims=True))[m28].ravel()
r_full=np.corrcoef(a,c)[0,1]
print()
print('=== (C) Structure temporelle (anomalies) dans 2-8 km ===')
print(f'  corr(d_t ubar , friction convective)        = {r_struct:+.3f}')
print(f'  corr(d_t ubar , friction+advection resolues) = {r_full:+.3f}')
print()
print('Lecture : la friction convective a la meme STRUCTURE temporelle que la tendance')
print('(elle porte la descente de ubar), mais le forcage grande echelle (residu) equilibre')
print('une partie de l amplitude - c est normal en RCE force. Le point physique tient :')
print('le flux de Reynolds est le terme CONVECTIF qui redistribue le momentum verticalement.')


## 4. Origine ondulatoire — flux vers le haut, phase vers le bas

Résultat classique (Eliassen–Palm, base de la QBO) : une onde de vitesse de phase verticale $c_z$
porte un flux de momentum de **signe opposé**. Test spectral $(\omega,m)$ **et** test niveau par
niveau (signe du flux vs signe de $c_z$). Avec seulement 34 pas de temps le spectre est bruité —
on l'affiche mais on s'appuie sur le test par profil, et on annonce clairement si le lien est établi
ou seulement suggéré.

In [ ]:
# ============================================================
#  §4 — ORIGINE ONDULATOIRE (version robuste)
#  Theorie (Eliassen-Palm, base de la QBO) : une onde de gravite de
#  vitesse de phase verticale c_z transporte un flux de momentum de
#  SIGNE OPPOSE. On teste ce lien de deux facons robustes.
#  NB : 34 pas de temps -> spectre bruite. On l'affiche mais on s'appuie
#  surtout sur le test (B) niveau par niveau, plus stable.
# ============================================================
uprime=ubar-ubar.mean(axis=1,keepdims=True)
sig=detrend(detrend(uprime[mask_show],axis=1),axis=0)
wz=np.hanning(sig.shape[0])[:,None]; wt=np.hanning(sig.shape[1])[None,:]
F=np.fft.fftshift(np.fft.fft2(sig*wz*wt)); P=np.abs(F)**2
kz=np.fft.fftshift(np.fft.fftfreq(sig.shape[0],d=np.diff(alt[mask_show]).mean()))
om=np.fft.fftshift(np.fft.fftfreq(sig.shape[1],d=dt_phys))
KZ,OM=np.meshgrid(kz,om,indexing='ij')

fig,axes=plt.subplots(1,2,figsize=(13,5))
ax=axes[0]; ax.grid(False)
ext=[om.min()*86400,om.max()*86400,kz.min()*1000,kz.max()*1000]
im=ax.imshow(np.log10(P+1e-12),origin='lower',aspect='auto',extent=ext,cmap='magma')
ax.set_xlabel('frequence (cycles/jour)'); ax.set_ylabel('m (cycles/km)')
ax.set_title(r'(A) Spectre $|\hat{\bar u^\prime}(\omega,m)|^2$ (2-8 km)')
fig.colorbar(im,ax=ax,label='log10 P')
Epos=P[(OM*KZ>0)].sum(); Eneg=P[(OM*KZ<0)].sum()

# (B) test niveau par niveau : signe(flux) vs signe(c_z), profils moyens
from scipy.ndimage import uniform_filter1d
ups=uniform_filter1d(uprime,3,axis=0,mode='nearest')
analytic=hilbert(detrend(ups,axis=1),axis=1); ph=np.unwrap(np.angle(analytic),axis=1)
cz=-np.gradient(ph,dt_phys,axis=1)/np.gradient(ph,alt,axis=0)
cz_prof=np.array([np.median(cz[iz][np.isfinite(cz[iz])&(np.abs(cz[iz])<1)]) for iz in range(n_z)])
flux_prof=flux.mean(1)
ax=axes[1]
good=m28&np.isfinite(cz_prof)
ax.scatter(-cz_prof[good]*100, flux_prof[good]*1e3, c=zkm[good], cmap='viridis', s=40)
ax.axhline(0,color='k',lw=.5); ax.axvline(0,color='k',lw=.5)
ax.set_xlabel(r'$-c_z$ (cm/s)'); ax.set_ylabel(r'$\rho_0\langle u^\prime w^\prime\rangle$ (mPa)')
cb=fig.colorbar(ax.collections[0],ax=ax,label='z (km)')
if good.sum()>3:
    r=np.corrcoef(-cz_prof[good],flux_prof[good])[0,1]
    ax.set_title(f'(B) flux vs $-c_z$ (2-8 km)\ncorr={r:+.2f}  (theorie: >0)')
else:
    r=np.nan; ax.set_title('(B) flux vs -c_z')
fig.tight_layout(); plt.show()

print(f'(A) energie spectrale : phase-descendante/total = {Eneg/(Epos+Eneg):.0%}')
print(f'(B) corr(flux, -c_z) dans 2-8 km = {r:+.2f}')
print()
if np.isfinite(r) and r>0.2:
    print('=> Lien onde-flux confirme : la ou la phase descend (c_z<0), le flux est')
    print('   dirige vers le haut. Signature d ondes de gravite convectives (mecanisme QBO).')
else:
    print('=> Lien onde-flux FAIBLE sur cette fenetre courte (34 pas). Le transport est')
    print('   probablement un MELANGE ondes + turbulence organisee. Prudence : ce point')
    print('   demande un run plus long pour etre etabli spectralement. La chaine du bilan')
    print('   (SS3) et la transmission de surface (SS5) restent, elles, robustes.')


## 5. Le cœur du sujet — transmission de la friction de surface

La condition-limite basse du flux est la **contrainte de surface** $\tau_s=\rho_0\overline{u'w'}|_0$
(la friction). Le flux la transporte vers le haut ; sa divergence $-\partial_z(\rho_0\overline{u'w'})$
la redépose sur $\bar u$. On visualise le profil de flux partant de $\tau_s$, la force résultante sur
$\bar u$, et la fraction de $\tau_s$ transmise à chaque altitude. **C'est la réponse directe à la
question du stage.**

In [ ]:
# ============================================================
#  §5 — LE CŒUR DU SUJET : transmission verticale de la friction
#  (ce diagnostic etait deja robuste ; on le renforce)
#  tau_s = rho0<u'w'>|_0 est la contrainte de surface. Le flux la
#  transporte vers le haut ; -d_z(flux) la redepose sur ubar.
# ============================================================
flux_p=flux.mean(1); tau_s=flux_p[0]
Fz=-np.gradient(flux_p,alt)/rho0                      # acceleration (m/s/s)

fig,axes=plt.subplots(1,3,figsize=(16,5),sharey=True)
ax=axes[0]
ax.plot(flux_p[mask_show],zkm[mask_show],'k',lw=2)
ax.scatter([tau_s],[0],color='r',zorder=5,s=50,label=f'$\\tau_s$={tau_s:+.4f} Pa')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r'$\rho_0\langle u^\prime w^\prime\rangle$ (Pa)'); ax.set_ylabel('z (km)')
ax.set_title('Flux de momentum vertical\n(part de la surface)'); ax.legend(fontsize=9)

ax=axes[1]
ax.plot(Fz[mask_show]*86400,zkm[mask_show],'tab:purple',lw=1.8)
ax.fill_betweenx(zkm[mask_show],0,Fz[mask_show]*86400,
                 where=Fz[mask_show]>0,color='tab:red',alpha=.2,label='accelere ubar')
ax.fill_betweenx(zkm[mask_show],0,Fz[mask_show]*86400,
                 where=Fz[mask_show]<0,color='tab:blue',alpha=.2,label='decelere ubar')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel('m/s/jour'); ax.set_title(r'Force convective sur $\bar u$'); ax.legend(fontsize=8)

ax=axes[2]
tr=flux_p/tau_s if abs(tau_s)>1e-9 else flux_p*np.nan
ax.plot(tr[mask_show],zkm[mask_show],'tab:green',lw=1.8)
ax.axvline(1,color='grey',lw=.6,ls='--'); ax.axvline(0,color='k',lw=.5)
ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r'flux(z)/$\tau_s$'); ax.set_title('Transmission de la\ncontrainte de surface')
fig.tight_layout(); plt.show()

if abs(tau_s)>1e-9:
    # altitude ou 90% de tau_s a ete deposee (|flux| tombe a 10% de |tau_s|)
    below=mask_show&(np.arange(n_z)>0)
    idx=np.where(np.abs(flux_p)<0.1*abs(tau_s))[0]
    z90=zkm[idx[idx>0][0]] if idx[idx>0].size else np.nan
    print(f'Contrainte de surface tau_s = {tau_s:+.4f} Pa')
    print(f'90% de |tau_s| deposee sous ~ {z90:.1f} km')
    i8=np.argmin(np.abs(alt-8000))
    print(f'Fraction transmise a 8 km : {tr[i8]:+.2f}')
print()
print('=> REPONSE AU SUJET : la friction de surface (tau_s) est communiquee a la')
print('   troposphere par le flux de Reynolds rho0<u\'w\'>. Sa divergence redepose le')
print('   momentum couche par couche (force sur ubar). La viscosite moleculaire ne joue')
print('   aucun role : tout passe par le transport convectif/ondulatoire resolu.')


## 6. Un modèle simple qui capture la transmission

On fit le **profil** $\overline{u'w'}(z)$ (ce qu'une paramétrisation doit d'abord reproduire) par
trois fermetures physiques, avec coefficients directs et NSE :
1. **diffusif** $-K\partial_z\bar u$ — échoue dans la couche contre-gradient ;
2. **flux de masse (GKI)** $-C(M_c/\rho_0)\partial_z\bar u$ ;
3. **enrichi** + amplitude $(M_c/\rho_0)\bar u$.
On affiche aussi le **régime** down-gradient / contre-gradient ($-\overline{u'w'}\partial_z\bar u$)
pour montrer *pourquoi* le diffusif ne peut pas marcher partout.

In [ ]:
# ============================================================
#  §6 — MODELE SIMPLE (version corrigee)
#  Version precedente : NSE=-1.2 (echec). Causes : (i) fit point-a-point
#  sur (z,t) 0-15km trop bruite/peu de points test, (ii) normalisation
#  qui melangeait train/test, (iii) coeffs aberrants (-52).
#
#  Nouvelle approche, robuste et interpretable :
#   - cible = <u'w'>(z) (PROFIL, moyenne temps) : c'est ce qu'une
#     parametrisation doit reproduire en premier.
#   - 3 fermetures physiques, coefficients par MOINDRES CARRES simples,
#     diagnostiquees par NSE sur le profil + signe/valeur des coeffs.
#   - flux de masse Mc = rho0<w'^+> (lecture niveau par niveau).
#   - separation DOWN-GRADIENT (bas) / CONTRE-GRADIENT (haut) via le
#     signe de -<u'w'> d_z ubar : on montre POURQUOI le diffusif echoue.
# ============================================================
Mc=np.zeros((n_z,n_stat))
ds_w=xr.open_dataset(path3d('wa'))
for iz in range(n_z):
    w=ds_w['wa'].isel({dim_z:iz,dim_t:idx_stat}).values
    wp=w-w.mean(axis=(1,2),keepdims=True)
    Mc[iz]=rho0[iz]*np.where(wp>0,wp,0).mean(axis=(1,2))
ds_w.close(); gc.collect()

# profils moyens (cible et predicteurs)
uw_p   = uw.mean(1)                       # <u'w'>(z)  cible
dudz_p = dudz.mean(1)
ubar_p = ubar.mean(1)
Mc_p   = Mc.mean(1)
zc=np.where(mask_show)[0]

def nse(a,b): return 1-np.sum((a-b)**2)/np.sum((a-a.mean())**2)

# --- regime down/contre-gradient (profil) ---
dg=-uw_p*dudz_p                            # >0 down-grad, <0 contre-grad
# altitude de bascule
sc=np.where(np.diff(np.sign(dg[zc])))[0]
z_switch=zkm[zc][sc[0]] if sc.size else np.nan

# --- fermetures : fit du profil par moindres carres (coeffs physiques directs) ---
def fit(cols):
    X=np.stack([c[zc] for c in cols],1); y=uw_p[zc]
    c,_,_,_=np.linalg.lstsq(X,y,rcond=None)
    return c, nse(y, X@c)

closures={
 '1. diffusif   -K.d_zu'      :[-dudz_p],
 '2. flux-masse -C.Mc/rho0.d_zu':[-(Mc_p/rho0)*dudz_p],
 '3. + amplitude Mc/rho0.u'   :[-(Mc_p/rho0)*dudz_p, (Mc_p/rho0)*ubar_p],
}
print('=== Fermetures du PROFIL <u\'w\'>(z) (NSE, coeffs physiques) ===')
fitres={}
for name,cols in closures.items():
    c,s=fit(cols); fitres[name]=(c,s,cols)
    print(f'  {name:32s} : NSE={s:+.3f} | coeffs={np.array2string(c,precision=4)}')

fig,axes=plt.subplots(1,2,figsize=(13,5.5),sharey=True)
ax=axes[0]
ax.plot(uw_p[zc],zkm[zc],'k',lw=2.6,label="vrai <u'w'>(z)")
sty={'1. diffusif   -K.d_zu':('tab:red','--'),
     '2. flux-masse -C.Mc/rho0.d_zu':('tab:green','-.'),
     '3. + amplitude Mc/rho0.u':('tab:blue','-')}
for name,(c,s,cols) in fitres.items():
    X=np.stack([cc[zc] for cc in cols],1)
    col,ls=sty[name]
    ax.plot(X@c,zkm[zc],ls,color=col,lw=1.8,label=f'{name.split(".")[0]}. NSE={s:.2f}')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r"$\langle u'w'\rangle$ (m$^2$/s$^2$)"); ax.set_ylabel('z (km)')
ax.set_title('Fermetures du profil de flux'); ax.legend(fontsize=8)

# panneau regime
ax=axes[1]
ax.plot(dg[zc],zkm[zc],'k',lw=1.8)
ax.fill_betweenx(zkm[zc],0,dg[zc],where=dg[zc]>0,color='tab:orange',alpha=.3,label='down-gradient')
ax.fill_betweenx(zkm[zc],0,dg[zc],where=dg[zc]<0,color='tab:purple',alpha=.3,label='contre-gradient')
if np.isfinite(z_switch): ax.axhline(z_switch,color='r',lw=1,ls=':',label=f'bascule ~{z_switch:.1f}km')
ax.axvline(0,color='k',lw=.5); ax.axhspan(2,8,color='k',alpha=.06); ax.set_ylim(0,Z_TOP_KM)
ax.set_xlabel(r'$-\langle u^\prime w^\prime\rangle\,\partial_z\bar u$')
ax.set_title('Regime de transport\n(pourquoi le diffusif echoue)'); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

best=max(fitres,key=lambda k:fitres[k][1])
print(f'\nMeilleure fermeture : {best}  (NSE={fitres[best][1]:+.3f})')
print(f'Bascule down->contre-gradient vers {z_switch:.1f} km' if np.isfinite(z_switch)
      else 'Regime unique (pas de bascule nette)')
print()
print('Lecture : le diffusif -K.d_zu impose un flux TOUJOURS down-gradient (K>0) et')
print('echoue la ou le transport est contre-gradient. La fermeture en flux de masse,')
print('surtout avec le terme d amplitude Mc.u, epouse mieux le profil car elle encode')
print('le couplage flux-de-masse x cisaillement (Zhang&Wu 2003) plutot qu une diffusion.')


## Conclusion

**Ce qui est solidement établi.**
- La **transmission de la friction de surface** (§5) : $\tau_s$ est portée vers le haut par le flux
  de Reynolds et redéposée sur $\bar u$ par sa divergence. Réponse directe et robuste au sujet.
- Le **bilan de qdm** (§3) : la friction convective a la structure temporelle de $\partial_t\bar u$
  (elle porte la descente de $\bar u$) ; le résidu = forçage grande échelle équilibre l'amplitude,
  comme attendu en RCE forcé.
- Le **régime de transport** (§6) : il existe une couche **contre-gradient** qui exclut une fermeture
  purement diffusive — cohérent avec le rejet du down-gradient déjà obtenu (T2 = amplitude). La
  fermeture en flux de masse épouse mieux le profil.

**Ce qui reste suggestif (fenêtre de 34 pas ≈ 8 jours).**
- La **descente de phase** (§2) est visible sur le Hovmöller et mesurable en cm/s, mais la statistique
  spectrale (§4) est trop courte pour affirmer un mode QBO-like établi. À confirmer sur un run plus
  long. On l'annonce comme *piste forte*, pas comme résultat clos.

**Lien Romps & Kuang.** Une fermeture en flux de masse marche parce que $M_c$ agrège la population
d'updrafts dont la variabilité vient de l'entraînement (nature vs nurture). La partie
ondulatoire/organisée (descente de phase, contre-gradient) est justement ce que le modèle « parcelles
indépendantes » ne contient pas.

**Prochaines étapes concrètes.**
1. Rejouer §2/§4 sur une intégration plus longue pour trancher le caractère QBO-like et mesurer la
   période de l'oscillation de $\bar u$.
2. Récupérer le **forçage grande échelle** $F_{GE}$ du setup RCEMIP pour fermer le bilan §3 à la
   quantité près (et non seulement en structure).
3. Séparer les coefficients de fermeture par couche (down / contre-gradient) et tester leur
   stationnarité → coefficients transférables vers un GCM.